GRADIENT BOOSTING NO PCA WITH LASSO FEATURES

Jack Silkaitis

In [ ]:
# Imports
import numpy as np
import pandas as pd
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.tree import DecisionTreeRegressor

from sklearn.metrics import mean_squared_error, mean_absolute_error
import plotly.express as px
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split, GridSearchCV, cross_validate
from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier
from sklearn.decomposition import PCA

In [18]:
# Import data
# df = pd.read_csv('./LassoFeatures.csv')
df = pd.read_csv('./LassoFeatures2.csv')
df.head(3)
df.shape

(1788, 80)

In [ ]:
# Cols
df.columns

Index(['Unnamed: 0', 'disc_year', 'release_month^2 sy_dist',
       'release_month^2 st_mass', 'release_month sy_pnum sy_w4mag',
       'release_month ttv_flag pl_tsystemref_BJD-UTC',
       'release_month elat pl_orbper', 'release_month glat pl_tsystemref_JD',
       'release_month st_nrvc sy_dist', 'sy_snum ttv_flag sy_w2mag',
       'sy_snum ttv_flag pl_tsystemref_BJD-UTC', 'sy_snum glon st_rad',
       'sy_snum sy_hmag^2', 'sy_snum sy_pmdec pl_tsystemref_BJD',
       'sy_pnum ttv_flag pl_orbper', 'sy_pnum elat st_rad',
       'sy_pnum elat sy_plx', 'sy_pnum glon sy_bmag', 'sy_pnum sy_pm sy_dist',
       'sy_pnum pl_orbper st_mass', 'sy_pnum pl_orbper pl_tsystemref_JD',
       'sy_pnum sy_bmag sy_plx', 'sy_pnum sy_w4mag^2',
       'ttv_flag^2 pl_tsystemref_BJD-UTC', 'ttv_flag pl_nnotes glon',
       'ttv_flag pl_nnotes st_teff',
       'ttv_flag pl_nnotes pl_tsystemref_BJD-UTC', 'ttv_flag elon glon',
       'ttv_flag release_year pl_tsystemref_BJD-UTC',
       'ttv_flag disc_year pl

In [21]:
# Split and prepare data
(df_train,df_test) = train_test_split(df,train_size=0.8,
                                      test_size=0.2,
                                      random_state=0)

In [ ]:
# Preprocess data
X_train = df_train.drop(['log radius','Unnamed: 0'],axis=1)
y_train = df_train['log radius']
X_test = df_test.drop(['log radius','Unnamed: 0'],axis=1)
y_test = df_test['log radius']

In [ ]:
# Standardize data
stnd = StandardScaler().set_output(transform='pandas')

X_number = X_train.select_dtypes(include='number')
X_categorical = X_train.select_dtypes(exclude='number')
X_number = stnd.fit_transform(X_number)
X_train = pd.concat([X_number, X_categorical], axis=1)

X_number = X_test.select_dtypes(include='number')
X_categorical = X_test.select_dtypes(exclude='number')
X_number = stnd.transform(X_number)
X_test = pd.concat([X_number, X_categorical], axis=1)

In [ ]:
# Baseline for comparison
tree = DecisionTreeRegressor(max_depth=2)
tree.fit(X_train,y_train)
tree.score(X_test, y_test)

0.19380744457122623

In [ ]:
# First CV
B = np.arange(1,301,100)
C = np.arange(1,10,3)
D = np.arange(0.01,1,0.33)
grid = {'n_estimators':B, 'max_depth':C, 'learning_rate':D}

gbt = GradientBoostingRegressor()
gbtCV = GridSearchCV(gbt,param_grid=grid,return_train_score=True,n_jobs=-1, verbose=2)
gbtCV.fit(X_train,y_train)

print('best params =',gbtCV.best_params_, '  valid acc =',gbtCV.best_score_)

Fitting 5 folds for each of 27 candidates, totalling 135 fits
best params = {'learning_rate': np.float64(0.34), 'max_depth': np.int64(1), 'n_estimators': np.int64(201)}   valid acc = 0.38142167415353545


In [ ]:
# Second CV
B = np.arange(150,300,50)
C = np.arange(1,5,1)
D = np.arange(.2, .5, .1)
grid = {'n_estimators':B, 'max_depth':C, 'learning_rate':D}

gbt = GradientBoostingRegressor()
gbtCV = GridSearchCV(gbt,param_grid=grid,return_train_score=True,n_jobs=-1, verbose=2)
gbtCV.fit(X_train,y_train)

print('best params =',gbtCV.best_params_, '  valid acc =',gbtCV.best_score_)

Fitting 5 folds for each of 36 candidates, totalling 180 fits
best params = {'learning_rate': np.float64(0.30000000000000004), 'max_depth': np.int64(1), 'n_estimators': np.int64(200)}   valid acc = 0.38889106256009043


In [ ]:
# Third CV
B = np.arange(150,250,25)
C = np.arange(.2,.45,.05)
grid = {'n_estimators':B, 'learning_rate':C}

gbt = GradientBoostingRegressor(max_depth=1)
gbtCV = GridSearchCV(gbt,param_grid=grid,return_train_score=True,n_jobs=-1, verbose=2)
gbtCV.fit(X_train,y_train)

print('best params =',gbtCV.best_params_, '  valid acc =',gbtCV.best_score_)

Fitting 5 folds for each of 20 candidates, totalling 100 fits
best params = {'learning_rate': np.float64(0.3), 'n_estimators': np.int64(200)}   valid acc = 0.3899676812717735


In [ ]:
# Best tree
bestGBT = GradientBoostingRegressor(max_depth=1, n_estimators=200, learning_rate=.3)

In [ ]:
# R2
print('Test R2',gbtCV.score(X_test,y_test))
print('Train R2',gbtCV.score(X_train,y_train))

Test R2 0.4515218221130428
Train R2 0.5968419732435852


In [ ]:
# Training MSE
y_pred = gbtCV.predict(X_train)
mean_squared_error(y_pred, y_train)

In [ ]:
#Runnable if data is lost
#bestGBT.fit(X_train, y_train)
# Testing R2
y_pred = gbtCV.predict(X_test)
mean_squared_error(y_pred, y_test)

0.039281330910366064

In [ ]:
# MAE
mean_absolute_error(y_pred,y_test)

0.15030621648697104

In [ ]:
# Mean rel err
np.mean(np.abs(10**(y_pred) - 10**y_test) / 10**y_test)

np.float64(0.35359992896250414)